#### State Persistance 

``` python
state = {
    "question": "What is LangGraph?",
    "answer": "",
    "messages": [],
    "retry_count": 0
}
```

Here The state contains all the information about the current Excecution.

#### State Persistance 
"Saving something so that it still exists even after the program stops"

instead of keeping the graph state only in RAM. Langgraph saves it somewhere.
Example: 
``` markdown
Graph excecution --> Current State --> Save Check point --> Database/disk--> Resume Later.
```

### Why we need State Persistance ?

* We need state persistence in LangGraph because it transitions large language model (LLM) applications from being stateless scripts into reliable, production-grade agentic systems. Without persistence, once a code execution ends, all context, message history, and variable updates are immediately wiped out.

* Persistence lets LangGraph applications keep useful information beyond a single graph run. It matters when an agent needs to continue a conversation, resume after an interruption, recover from a failure, or remember information across interactions.

* LangGraph provides two complementary persistence systems:
    * Checkpointers persist a thread’s graph state as checkpoints. Use them for short-term, thread-scoped memory, including conversation continuity, human-in-the-loop workflows, time travel, and fault tolerance.
    * Stores persist application-defined data outside the graph state. Use them for long-term, cross-thread memory, including user preferences, facts, and shared knowledge.
* Most applications can use both: a checkpointer tracks the current thread, and a store tracks durable information across threads.

##### What is LangGraph Persistence?
Persistence refers to ability to save and restore the state of workflow over the time.

langgraph has built-in persistance layer
* it uses checkpointers to save the graphs state
* Every super-step(every node) in graph can be saved automatically

Checkpoints are stored in threads.
* After the Graph runs, we can access it state from the thread.
* This makes the graph reusable and inspectable

Cool things you can do with this.

* Human-in-the-loop: Humans can jump in, check, or change the graph while it runs.
* Memory: The graph “remembers” past states.
* Time travel: You can go back to any previous checkpoint and see the state.
* Fault-tolerance: If something goes wrong, you can resume from a saved checkpoint instead of starting over.

Explore this from every angle.
#### What are Threads

Threads are like separate "conversations" or runs of your graph. each thread has a unique thread_id and keeps its own set of checkpoints, so its excecution history says separate and independednt from other threads.

Think of threads as separate chat conversations — each maintains its own state and history, independent of others.

A thread’s current and historical state can be retrieved. To persist state, a thread must be created prior to executing a run.

Example of using a thread:
``` python
# Every graph invocation requires a thread_id
config = {"configurable": {"thread_id": "user_session_123"}}
result = graph.invoke(input_data, config)
```


#### What are Checkpoints?

Every time your graph executes a step (called a “super-step”), LangGraph automatically creates a checkpoint — a complete snapshot of your graph’s state at that moment. Each checkpoint contains:

* Values: The current state of all channels
* Next nodes: Which nodes should execute next
* Tasks: Pending operations and any error information
* Metadata: Execution context and timing information
* Config: Thread and checkpoint identifiers

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict


# Define the state
class State(TypedDict):
    message: str
    steps: list[str]

# Define simple nodes
def start_node(state: State):
    return {"message": "Hello", "steps": ["start"]}

def middle_node(state: State):
    return {"message": state["message"] + " -> Middle", "steps": state["steps"] + ["middle"]}

def end_node(state: State):
    return {"message": state["message"] + " -> End", "steps": state["steps"] + ["end"]}

# Create the workflow
workflow = StateGraph(State)
workflow.add_node(start_node)
workflow.add_node(middle_node)
workflow.add_node(end_node)

workflow.add_edge(START, "start_node")
workflow.add_edge("start_node", "middle_node")
workflow.add_edge("middle_node", "end_node")
workflow.add_edge("end_node", END)

# Set up checkpointing
checkpointer = InMemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

# Run the graph
graph.invoke({"message": "", "steps": []}, config={"configurable": {"thread_id": "1"}})

# Access checkpoint data
print(checkpointer.get_thread("1"))

AttributeError: 'InMemorySaver' object has no attribute 'get_thread'

LangGraph’s InMemorySaver checkpointer saves a checkpoint at every super-step, which generally means after each node execution.

* start_node → checkpoint 1
* middle_node → checkpoint 2
* end_node → checkpoint 3

✅ So after running the graph, you should have 3 checkpoints in memory.

#### How to retrieving and inspecting state?

* Current State

Access the latest state of any thread instantly by calling graph.get_state(config).

``` python
config = {"configurable": {"thread_id": "workflow_123"}}
current_state = graph.get_state(config)

print(f"Current values: {current_state.values}")
print(f"Next nodes: {current_state.next}")
print(f"Step: {current_state.metadata['step']}")
```

* ◉ Historical Journey

View the complete execution history by calling ```graph.get_state_history(config)```.
``` python
config = {"configurable": {"thread_id": "workflow_123"}}
history = list(graph.get_state_history(config))

# History is ordered from most recent to oldest
for i, checkpoint in enumerate(history):
    print(f"Step {i}: {checkpoint.values}")
    print(f"Next: {checkpoint.next}")
    print("---")
```

#### What are Real-World Use Cases?

1. Human-in-the-Loop Approval

Pauses graph execution until human approves.

``` python
def requires_approval(state):
    if state.get("requires_human_review"):
        # Graph pauses; checkpoint stored here
        return {"status": "pending_approval"}
    return {"status": "approved"}

# Human can later update state externally
# graph.update_state(config, {"requires_human_review": False, "status": "approved"})
````


2. Multi-Session Customer Support

Each customer gets a persistent thread with memory.

``` python
customer_config = {"configurable": {
    "thread_id": f"customer_{customer_id}_session_{session_id}",
    "user_id": customer_id
}}

# All conversations in this thread retain memory across sessions
graph.run("Hello, I need help", config=customer_config)
```



3. Fault-Tolerant Data Processing

Checkpointing ensures safe restart from last success.

``` python
from langgraph import Graph

graph = Graph()

def risky_task(_):
    result = numerator / denominator  
    return {"result": result}

try:
    graph.run(risky_task)
except Exception:
    # Resume execution from last checkpoint
    graph.resume(checkpoint_id="last_good_step")
```

#### State Persistence Components

State Persistence

``` markdown
├── Checkpoints
├── Memory
├── Saving State
├── Resuming Execution
├── Durable Execution
└── Thread Management
```

1. CheckPoints

A check point is the simply ```snapshot of the graph state at particular moment.```

Analogy : imagine your playing a vedio game. Game autosaves, tommorrow, you continue from level 10, not level 1 that saving point is a checkpoint.

A checkpoint is a snapshot of the entire graph state saved after every node executes. LangGraph writes it automatically whenever you attach a checkpointer. Think of it as an auto-save slot that captures exactly where the graph is at any point in time.


Like a video game that saves progress after every level. If the game crashes, you reload from the last checkpoint — you don't restart from the beginning.


MemorySaver dev / testing
* Stores checkpoints in Python RAM. Fast, zero setup. Lost on process restart. Use during development.
* SqliteSaver production
Persists checkpoints to a SQLite file. Survives restarts. Good for single-server deployments.

##### langGraph Example
``` markdown
start --> read pdf --> OCR --> Extract data --> LLM Analysis --> End
```
Suppose after OCR -> Langgraph Saves 

``` python
state = {
    "document":"...",
    "ocr_text":"...",
    "page: 12
} # this is a check point
```

``` markdown
start --> read pdf --> OCR --> ✔ Checkpoint Saved--> Extract data --> LLM Analysis --> End
```

Suppose if the programm crashes in after ocr it will restrat from checkpoint --> extract data. not from the START.

Langgraph provides Checkpoint savers.

``` python
from langgraph.checkpoint.memory import MemorySaver
memory = MemorySaver()

# now pass the above memory while compiling
app = builder.compile(checkpointer = memory)
```

Now the graph automatically saves checkpoints.


In [ ]:
# Real-world: customer support bot that saves after every step
from langgraph.graph import StateGraph, END # type:ignore
from langgraph.checkpoint.memory import MemorySaver # type:ignore
from langgraph.checkpoint.sqlite import SqliteSaver # type:ignore
from typing import TypedDict, List

class SupportState(TypedDict):
    ticket_id: str
    messages: List[str]
    category: str
    resolved: bool

def classify(state):
    return {"category": "billing"}

llm = ""
def respond(state):
    reply = llm.invoke(state["messages"])
    return {"messages": state["messages"] + [reply], "resolved": True}

builder = StateGraph(SupportState)
builder.add_node("classify", classify)
builder.add_node("respond", respond)
builder.set_entry_point("classify")
builder.add_edge("classify", "respond")
builder.add_edge("respond", END)

# DEV: in-memory checkpointer
memory = MemorySaver()
graph = builder.compile(checkpointer=memory)

# PROD: SQLite checkpointer (persists to disk)
with SqliteSaver.from_conn_string("checkpoints.db") as db:
    graph = builder.compile(checkpointer=db)


# Every invoke auto-saves a checkpoint after each node
result = graph.invoke(
    {"ticket_id": "TKT-001", "messages": ["My invoice is wrong"]},
    config={"configurable": {"thread_id": "TKT-001"}}
)


What gets saved in each checkpoint

* The full state dict (all keys and values at that moment)
* Which node just ran and which node is next
* A timestamp and a unique checkpoint ID
* The thread_id so checkpoints from different conversations don't mix

#### Memory

LangGraph has two kinds of memory. 

Short-term memory lives in the state dict and disappears when the graph ends.

Long-term memory is stored in the checkpointer and persists across separate invocations of the same thread.

Short-term memory
* The messages list in state. Lives only for this single graph run. Gone when .invoke() returns.

Long-term memory
* Anything written to the checkpointer. Survives restarts. Retrieved the next time you call the same thread_id.

In [ ]:
# Real-world: multi-turn sales assistant that remembers context
from langchain_core.messages import HumanMessage, AIMessage # type:ignore
from langgraph.checkpoint.sqlite import SqliteSaver # type:ignore

class SalesState(TypedDict):
    messages: List          # short-term: grows each turn
    customer_name: str      # long-term: remembered across sessions
    product_interest: str   # long-term: e.g. "Enterprise plan"
    deals_discussed: List   # long-term: running list of deals raised

def chat_node(state):
    # LLM has access to FULL history because messages is in state
    response = llm.invoke(state["messages"])
    return {"messages": state["messages"] + [AIMessage(response)]}

graph = builder.compile(checkpointer=SqliteSaver.from_conn_string("sales.db"))
config = {"configurable": {"thread_id": "customer-anand-sharma"}}

# --- Session 1 (Monday) ---
graph.invoke({
    "messages": [HumanMessage("I'm interested in your Enterprise plan")],
    "customer_name": "Anand Sharma",
    "product_interest": "Enterprise",
    "deals_discussed": []
}, config=config)

# --- Session 2 (Wednesday) — new Python process, same thread_id ---
# LangGraph loads the checkpoint from session 1 automatically!
graph.invoke({
    "messages": [HumanMessage("What's the pricing for 50 seats?")]
    # No need to re-send customer_name or product_interest —
    # they are already in the loaded checkpoint
}, config=config)

When you call graph.invoke() with a thread_id that already has checkpoints, LangGraph automatically loads the last saved state and merges your new input into it. You get the full history for free.


In [ ]:
# Reading memory directly — inspect what the graph remembers
current_state = graph.get_state(config)
print(current_state.values["customer_name"])      # "Anand Sharma"
print(current_state.values["product_interest"])   # "Enterprise"
print(len(current_state.values["messages"]))      # total turns so far

# View the full checkpoint history for a thread
for checkpoint in graph.get_state_history(config):
    print(checkpoint.config["configurable"]["checkpoint_id"])
    print(checkpoint.metadata["step"])   # which step this was

#### Saving state

State is saved automatically after every node — but you can also write to it manually with update_state(). This lets you inject data from outside the graph, correct mistakes, or add context mid-run.

Like a doctor adding notes to a patient record during a visit. The record is auto-updated after each examination, but the doctor can also manually annotate it at any time.

In [ ]:
# Real-world: document review pipeline
# A human reviewer edits the AI's draft before it continues

class ReviewState(TypedDict):
    document_text: str
    ai_summary: str
    human_edits: str
    final_approved: bool
    reviewer_name: str

def summarise(state):
    summary = llm.invoke(f"Summarise: {state['document_text']}")
    return {"ai_summary": summary}

def finalise(state):
    # Uses human_edits if present, otherwise uses ai_summary
    final = state.get("human_edits") or state["ai_summary"]
    return {"final_approved": True}

graph = builder.compile(
    checkpointer=MemorySaver(),
    interrupt_after=["summarise"]  # pause after AI drafts
)
config = {"configurable": {"thread_id": "review-doc-42"}}


# Step 1: Run until interrupt
graph.invoke({"document_text": "Q3 sales grew 34% YoY..."}, config=config)

# Step 2: Human reads the AI summary and corrects it
state = graph.get_state(config)
print("AI draft:", state.values["ai_summary"])

# Step 3: Manually write corrected data into the checkpoint
graph.update_state(
    config,
    {
        "human_edits": "Q3 sales grew 34% YoY, driven by APAC expansion",
        "reviewer_name": "Priya Menon"
    },
    as_node="summarise"   # treat this update as if summarise node wrote it
)
# Step 4: Resume — finalise node now sees the human edits
graph.invoke(None, config=config)


#### update_state() — key parameters
1. config — which thread to update (must have a checkpointer)
2. values — a dict of state keys to overwrite. Merged into current state, not a full replace.
3. as_node — optional. Tells the checkpointer which node "wrote" this update, so the next node in the graph runs correctly after it.



#### Resuming execution
A graph that was interrupted (by a breakpoint, a crash, or a deliberate pause) can be resumed from the exact checkpoint where it stopped. You just call .invoke(None, config=...) again with the same thread_id.

Like pressing "Resume" on a paused download. The browser doesn't re-download the first 80% — it picks up exactly from byte 81.

In [ ]:
# Real-world: multi-step loan application
# Steps: intake → credit_check → risk_score → human_approval → disburse
# If credit bureau times out, we resume when it's back up

class LoanState(TypedDict):
    application_id: str
    applicant: dict
    credit_score: int
    risk_level: str
    approved: bool
    disbursed: bool

credit_bureau = ""
def credit_check(state):
    try:
        score = credit_bureau.lookup(state["applicant"]["pan"])
        return {"credit_score": score}
    except TimeoutError:
        # Raise — this will save checkpoint BEFORE this node
        raise   # graph halts here; state up to this point is saved

graph = builder.compile(checkpointer=SqliteSaver.from_conn_string("loans.db"))
config = {"configurable": {"thread_id": "LOAN-2024-8821"}}

# --- Attempt 1: credit bureau times out ---
try:
    graph.invoke({
        "application_id": "LOAN-2024-8821",
        "applicant": {"name": "Ravi Kumar", "pan": "ABCDE1234F"}
    }, config=config)
except TimeoutError:
    print("Credit bureau offline — will retry later")

# --- 30 minutes later: credit bureau is back ---
# Resume from exact checkpoint — intake node is NOT re-run
result = graph.invoke(None, config=config)
print("Loan processed:", result["disbursed"])

In [ ]:
# Resume with updated input — change something before continuing
# Scenario: human reviewer rejects the risk score and overrides it

# 1. Check current state
state = graph.get_state(config)
print("Current risk level:", state.values["risk_level"])   # "HIGH"

# 2. Override before resuming
graph.update_state(config, {"risk_level": "MEDIUM"}, as_node="risk_score")

# 3. Resume — the next node (human_approval) sees MEDIUM, not HIGH
graph.invoke(None, config=config)

# Replay from a SPECIFIC past checkpoint (time-travel debugging)
history = list(graph.get_state_history(config))
old_checkpoint = history[-3]   # 3 steps back
graph.invoke(None, config=old_checkpoint.config)   # re-runs from there


The last line is time-travel: you can replay execution from any historical checkpoint. This is invaluable for debugging agent failures — go back in time, fix the state, and re-run forward.


##### 5. Durable Excecution

Durable execution means the graph can survive crashes, server restarts, and long waits — because every step is checkpointed. Even if your server goes down mid-run, the next restart picks up from the last saved node.

Like a GPS rerouting after you exit the highway. It doesn't restart your whole journey — it figures out where you are now and continues from there.


Without a checkpointer, if your Python process dies mid-graph, everything is lost. With a persistent checkpointer (SQLite or PostgreSQL), the graph is resumable from any restart.


In [ ]:
# Real-world: nightly data pipeline that can take 2+ hours
# Steps: extract → validate → transform → load → notify
# If the server reboots during transform, we resume from there

from langgraph.checkpoint.sqlite import SqliteSaver # type:ignore
import time

class PipelineState(TypedDict):
    run_id: str
    source: str
    raw_rows: int
    valid_rows: int
    transformed_rows: int
    loaded: bool
    errors: List[str]

def extract(state):
    print(f"Extracting from {state['source']}...")
    rows = db.read_all(state["source"])
    return {"raw_rows": len(rows)}

def validate(state):
    valid = [r for r in get_rows() if is_valid(r)]
    return {"valid_rows": len(valid)}

def transform(state):
    # Expensive — takes 90 minutes. Durable = if we crash here,
    # we resume from this node on next start, not from extract.
    result = run_heavy_transform(state["valid_rows"])
    return {"transformed_rows": result}

ef load(state): ...
def notify(state): ...

with SqliteSaver.from_conn_string("pipeline.db") as db_cp:
    graph = builder.compile(checkpointer=db_cp)
    config = {"configurable": {"thread_id": "nightly-2024-07-08"}}

    # On first start: runs extract → validate → transform → load → notify
    # On restart after crash: skips already-completed nodes,
    # continues from last checkpoint automatically
    result = graph.invoke(
        {"run_id": "nightly-2024-07-08", "source": "sales_db"},
        config=config
    )

#### Thread management

A thread is an isolated, named conversation or workflow run. Every call with the same thread_id shares state. Different thread_ids are completely independent — like separate browser tabs each with their own session.

Like WhatsApp chats. Each conversation (thread) has its own history. You don't want your conversation with your boss mixing with your family group. Same graph, totally separate state.



In [ ]:
# Real-world: multi-tenant SaaS chatbot
# Each user (and each support ticket) gets their own thread

class ChatState(TypedDict):
    messages: List
    user_id: str
    subscription_tier: str
    tokens_used: int

def respond(state):
    reply = llm.invoke(state["messages"])
    return {
        "messages": state["messages"] + [reply],
        "tokens_used": state["tokens_used"] + count_tokens(reply)
    }

graph = builder.compile(checkpointer=SqliteSaver.from_conn_string("chats.db"))

# Thread per USER — remembers preferences and full history
def user_config(user_id: str):
    return {"configurable": {"thread_id": f"user-{user_id}"}}

# Thread per TICKET — isolated to one support case
def ticket_config(ticket_id: str):
    return {"configurable": {"thread_id": f"ticket-{ticket_id}"}}

# User 1 asks two questions across two days — state is shared
graph.invoke({"messages": ["What's my bill?"], "user_id": "U001",
              "subscription_tier": "pro", "tokens_used": 0},
             config=user_config("U001"))

graph.invoke({"messages": ["And what's my renewal date?"]},   # state loaded!
             config=user_config("U001"))

# User 2 is completely isolated — no cross-contamination
graph.invoke({"messages": ["Cancel my subscription"], "user_id": "U002",
              "subscription_tier": "free", "tokens_used": 0},
             config=user_config("U002"))


In [ ]:
# Thread management operations
# Get current state of a thread
state = graph.get_state(user_config("U001"))
print("Tokens used:", state.values["tokens_used"])

# List ALL checkpoints for a thread (full audit trail)
for snapshot in graph.get_state_history(user_config("U001")):
    print(snapshot.created_at, snapshot.metadata["step"])

# Delete a thread (GDPR right to erasure, for example)
from langgraph.checkpoint.base import BaseCheckpointSaver
checkpointer.delete_thread("user-U001")

# Use sub-threads for branching within one session
# e.g. A/B testing two response strategies for the same user query
variant_a = {"configurable": {"thread_id": "user-U001-variant-a"}}
variant_b = {"configurable": {"thread_id": "user-U001-variant-b"}}

graph.invoke({"messages": ["Upgrade me"]}, config=variant_a)
graph.invoke({"messages": ["Upgrade me"]}, config=variant_b)


Thread ID naming conventions

* user-{user_id} — one thread per user for conversational memory
* ticket-{ticket_id} — one thread per support case, deleted when closed
* run-{date}-{pipeline} — one thread per ETL/pipeline run for durability
* session-{uuid} — ephemeral, one per browser session, expires on logout